# 03. Baseline Models

Xem lại kết quả của majority classifier, TF-IDF word SVM và TF-IDF char SVM.

## 1. Setup

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, Image

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd

print("Project root:", ROOT)

TABLES_DIR = ROOT / "outputs" / "tables"
FIGURES_DIR = ROOT / "outputs" / "figures"
PREDICTIONS_DIR = ROOT / "outputs" / "predictions"
MODELS_DIR = ROOT / "outputs" / "models" / "baseline"
REPORTS_DIR = ROOT / "outputs" / "reports"

## 2. Kiểm tra file output bắt buộc

In [ ]:
required_files = {
    "baseline_results": TABLES_DIR / "baseline_results.csv",
    "baseline_classification_report": TABLES_DIR / "baseline_classification_report.csv",
    "baseline_predictions": PREDICTIONS_DIR / "baseline_predictions.csv",
    "baseline_report": REPORTS_DIR / "baseline_report.md",
}

check_df = pd.DataFrame(
    [{"name": name, "path": str(path), "exists": path.exists()} for name, path in required_files.items()]
)

display(check_df)

missing = check_df.loc[~check_df["exists"], "name"].tolist()
if missing:
    raise FileNotFoundError(f"Missing required baseline files: {missing}")

print("All required Stage 3 files exist.")

## 3. Kết quả tổng hợp baseline

Metric chính là **Macro-F1** vì EDA cho thấy cả sentiment và topic đều mất cân bằng lớp.

In [ ]:
baseline_results = pd.read_csv(TABLES_DIR / "baseline_results.csv")
display(baseline_results)

## 4. Kết quả trên test set

Bảng này là mốc chính để so sánh với PhoBERT ở các giai đoạn sau.

In [ ]:
test_results = baseline_results[baseline_results["split"] == "test"].copy()
test_results = test_results.sort_values(["task", "macro_f1"], ascending=[True, False])
display(test_results)

best_by_task = (
    test_results.sort_values(["task", "macro_f1"], ascending=[True, False])
    .groupby("task")
    .head(1)
    .reset_index(drop=True)
)

display(Markdown("### Best baseline by Macro-F1 on test set"))
display(best_by_task)

## 5. Sentiment baseline analysis

In [ ]:
sentiment_test = test_results[test_results["task"] == "sentiment"].copy()
display(sentiment_test)

best_sentiment = sentiment_test.sort_values("macro_f1", ascending=False).iloc[0]
display(Markdown(
    f"Best sentiment baseline by Macro-F1: **{best_sentiment['model']}** "
    f"(Macro-F1 = **{best_sentiment['macro_f1']:.4f}**, "
    f"Accuracy = **{best_sentiment['accuracy']:.4f}**)."
))

### Nhận xét sentiment

Ghi nhận chính:

```text
- Majority baseline có Accuracy không quá thấp nhưng Macro-F1 rất thấp.
- Điều này xác nhận Accuracy không đủ đáng tin trên dữ liệu mất cân bằng.
- TF-IDF word SVM thường có Accuracy cao.
- TF-IDF char SVM có thể cải thiện Macro-F1 nhờ xử lý tốt hơn lớp nhỏ neutral.
```

Khi so sánh PhoBERT cho sentiment, mốc cần vượt là baseline có Macro-F1 cao nhất trên test set.

## 6. Topic baseline analysis

In [ ]:
topic_test = test_results[test_results["task"] == "topic"].copy()
display(topic_test)

best_topic = topic_test.sort_values("macro_f1", ascending=False).iloc[0]
display(Markdown(
    f"Best topic baseline by Macro-F1: **{best_topic['model']}** "
    f"(Macro-F1 = **{best_topic['macro_f1']:.4f}**, "
    f"Accuracy = **{best_topic['accuracy']:.4f}**)."
))

### Nhận xét topic

Ghi nhận chính:

```text
- Majority baseline topic có Accuracy cao vì lớp lecturer chiếm đa số.
- Nhưng Macro-F1 rất thấp, chứng minh model không học được các lớp nhỏ.
- TF-IDF word SVM là baseline mạnh cho topic nếu có Macro-F1 cao nhất.
- Lớp others cần phân tích kỹ vì đây thường là lớp rộng và mơ hồ.
```

Khi so sánh PhoBERT cho topic, không chỉ xem Accuracy. Phải xem Macro-F1 và per-class F1.

## 7. Classification report

Phần này kiểm tra chi tiết Precision, Recall, F1 theo từng lớp.

In [ ]:
classification_report_df = pd.read_csv(TABLES_DIR / "baseline_classification_report.csv")
display(classification_report_df.head(20))

## 8. Per-class F1 trên test set

Tập trung vào các lớp nhỏ:

```text
Sentiment:
- neutral

Topic:
- facility
- others
```

In [ ]:
test_class_report = classification_report_df[
    (classification_report_df["split"] == "test")
].copy()

# Loại bỏ các dòng tổng hợp để xem nhãn thật
summary_labels = {"accuracy", "macro avg", "weighted avg"}
per_class_test = test_class_report[~test_class_report["label"].isin(summary_labels)].copy()

display(per_class_test.sort_values(["task", "model", "f1_score"]))

display(Markdown("### Lowest per-class F1 cases"))
display(per_class_test.sort_values("f1_score").head(10))

## 9. Confusion matrices

Các hình dưới đây giúp kiểm tra model nhầm lớp nào sang lớp nào. Với dữ liệu mất cân bằng, confusion matrix rất quan trọng.

In [ ]:
def show_confusion_matrix(task: str, model: str, split: str = "test"):
    path = FIGURES_DIR / f"confusion_matrix_baseline_{task}_{model}_{split}.png"
    if not path.exists():
        display(Markdown(f"Missing confusion matrix: `{path}`"))
        return
    display(Markdown(f"### {task} — {model} — {split}"))
    display(Image(filename=str(path)))

for task in ["sentiment", "topic"]:
    for model in ["majority", "tfidf_word_svm", "tfidf_char_svm"]:
        show_confusion_matrix(task, model, split="test")

## 10. Validation vs Test stability

Nội dung: kiểm tra kết quả validation và test có chênh lệch bất thường không.

In [ ]:
stability = baseline_results.pivot_table(
    index=["task", "model"],
    columns="split",
    values="macro_f1"
).reset_index()

if "validation" in stability.columns and "test" in stability.columns:
    stability["val_test_macro_f1_gap"] = stability["validation"] - stability["test"]

display(stability)

## 11. Baseline conclusions

Kết luận cần ghi cho báo cáo:

```text
1. Majority baseline cho thấy cả hai task bị ảnh hưởng bởi mất cân bằng lớp.
2. Accuracy không đủ để đánh giá, đặc biệt với topic vì lecturer chiếm đa số.
3. TF-IDF/SVM vượt xa Majority, nên đây là baseline hợp lý trước khi dùng PhoBERT.
4. Với sentiment, cần chú ý lớp neutral vì đây là lớp nhỏ và dễ bị bỏ qua.
5. Với topic, cần chú ý facility và others, đặc biệt others vì bản chất lớp này rộng và mơ hồ.
6. PhoBERT ở giai đoạn sau phải được so sánh chủ yếu bằng Macro-F1 và per-class F1.
```

Baseline chính để so sánh với PhoBERT:

```text
Sentiment:
- Chọn model có Macro-F1 test cao nhất.

Topic:
- Chọn model có Macro-F1 test cao nhất.
```

In [ ]:
display(Markdown("### Final baseline reference"))
for _, row in best_by_task.iterrows():
    display(Markdown(
        f"- **{row['task']}**: `{row['model']}` "
        f"with Macro-F1 = **{row['macro_f1']:.4f}**, "
        f"Accuracy = **{row['accuracy']:.4f}**"
    ))

## 12. Stage 3 status

Giai đoạn Baseline hoàn thành khi có đủ:

```text
- baseline_results.csv
- baseline_classification_report.csv
- baseline_predictions.csv
- confusion_matrix_baseline_*.png
- baseline_report.md
- baseline model artifacts trong outputs/models/baseline/
```

Giai đoạn tiếp theo đề xuất:

```text
Stage 4 — Noisy Test Generation
```

Lý do làm noisy test trước PhoBERT:

```text
- Đóng gói sẵn toàn bộ dữ liệu clean/noisy trước khi đưa lên Kaggle.
- Baseline có thể evaluate lại trên noisy test ở local.
- PhoBERT sau đó chỉ cần train/evaluate trên Kaggle theo cùng dữ liệu.
```